# Algorithmic Trading with Python & Google Colab
## Session 2: Teaching a Neural Network to Trade: GPU-Accelerated Deep Learning

*(c) Dr. Yves J. Hilpisch | The Python Quants GmbH | https://tpq.io | https://hilpisch.com*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yveshilpisch/pyalgo/blob/main/notebooks/02_deep_learning_gpu_trading.ipynb)

---

### Objectives
1. **GPU Hardware Acceleration**: Detecting CUDA devices and benchmarking tensor matrix multiplication throughput.
2. **Quantitative Feature Engineering**: Constructing lagged return features, rolling volatility, and momentum indicators.
3. **PyTorch Model Pipeline**: Implementing custom `Dataset`, `DataLoader`, and multi-layer `TradingDNN` architectures.
4. **GPU-Accelerated Training**: Optimizing cross-entropy loss with AdamW, learning rate schedules, and model checkpointing.
5. **Loss vs. Economic Alpha**: Statistical accuracy vs. trading profitability; filtering signals with confidence thresholds.
6. **The Overfitting Experiment**: Analyzing model capacity (Tiny vs. Balanced vs. Oversized networks).


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Detect Hardware Accelerator
if torch.cuda.is_available():
    device = torch.device('cuda')
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_name = "Apple Silicon MPS"
else:
    device = torch.device('cpu')
    device_name = "CPU (Colab standard runtime)"

print(f"Active Compute Device: {device} ({device_name})")


## 1. Hardware Acceleration Benchmark: CPU vs. GPU

We benchmark tensor matrix multiplications on CPU vs. GPU.


In [ ]:
size = 4000
# CPU Benchmark
x_cpu = torch.randn(size, size, device='cpu')
t0 = time.time()
_ = torch.matmul(x_cpu, x_cpu)
t_cpu = time.time() - t0

# GPU / Accelerator Benchmark
if device.type != 'cpu':
    x_dev = torch.randn(size, size, device=device)
    _ = torch.matmul(x_dev, x_dev) # warmup
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(10):
        _ = torch.matmul(x_dev, x_dev)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_dev = (time.time() - t0) / 10
    speedup = t_cpu / t_dev
    print(f"Matrix ({size}x{size}) -> CPU: {t_cpu:.4f}s | {device_name}: {t_dev:.4f}s (Speedup: {speedup:.1f}x)")
else:
    print(f"Matrix ({size}x{size}) -> CPU: {t_cpu:.4f}s (Running on CPU runtime)")


## 2. Quantitative Feature Pipeline

We create a rich feature set $X_t$:
- Lagged log-returns ($lag_1, \dots, lag_5$).
- Rolling return volatility (20-day standard deviation).
- Rolling return momentum (10-day moving average).
- Binary directional target $y_t \in \{0, 1\}$.


In [ ]:
DATA_URL = "https://hilpisch.com/eod_data.csv"
df = pd.read_csv(DATA_URL, parse_dates=['Date']).set_index('Date').sort_index()
symbol = 'SPY'
prices = df[symbol].dropna()

# Feature construction (using canonical ordering)
features_df = pd.DataFrame(index=prices.index)
features_df['price'] = prices
features_df['return'] = np.log(prices / prices.shift(1))

feature_cols = []
for lag in range(1, 6):
    col = f'lag_{lag}'
    features_df[col] = features_df['return'].shift(lag)
    feature_cols.append(col)

features_df['vol_20'] = features_df['return'].shift(1).rolling(20, min_periods=20).std(ddof=1)
features_df['mom_10'] = features_df['return'].shift(1).rolling(10, min_periods=10).mean()
feature_cols.extend(['vol_20', 'mom_10'])

features_df['target_return'] = features_df['return']
features_df['target_dir'] = (features_df['target_return'] > 0).astype(int)

features_df = features_df.dropna()
print(f"Engineered Features ({len(feature_cols)}): {feature_cols}")
print(f"Total Clean Samples: {len(features_df)}")


## 3. Dataset Splitting & PyTorch DataLoaders

We split chronologically (60% Train, 20% Validation, 20% Test) and scale features using `StandardScaler` fit strictly on train data.


In [ ]:
n = len(features_df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = features_df.iloc[:train_end]
val_df = features_df.iloc[train_end:val_end]
test_df = features_df.iloc[val_end:]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])

y_train = train_df['target_dir'].values
y_val = val_df['target_dir'].values
y_test = test_df['target_dir'].values

class FinancialDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(FinancialDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(FinancialDataset(X_val, y_val), batch_size=64, shuffle=False)
test_loader = DataLoader(FinancialDataset(X_test, y_test), batch_size=64, shuffle=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 4. Deep Neural Network Architecture (`TradingDNN`)

We define a multi-layer deep network with Batch Normalization and Dropout.


In [ ]:
class TradingDNN(nn.Module):
    def __init__(self, input_dim, hidden_units=[64, 32], dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_units:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

model = TradingDNN(input_dim=len(feature_cols), hidden_units=[64, 32], dropout=0.2).to(device)
print(model)


## 5. GPU Training Pipeline & Model Checkpointing

We train with AdamW and save both the model weights and the fitted feature normalization parameters (`scaler.mean_`, `scaler.scale_`) to ensure exact inference parity in Session 3.


In [ ]:
epochs = 80
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float('inf')
model_path = 'best_trading_dnn.pt'

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * len(by)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == by).sum().item()
        total += len(by)
        
    t_loss = running_loss / total
    t_acc = correct / total
    train_losses.append(t_loss)
    train_accs.append(t_acc)
    
    # Validation
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            v_loss += loss.item() * len(by)
            preds = (torch.sigmoid(logits) > 0.5).float()
            v_corr += (preds == by).sum().item()
            v_tot += len(by)
            
    val_loss = v_loss / v_tot
    val_acc = v_corr / v_tot
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            "state_dict": model.state_dict(),
            "scaler_mean": scaler.mean_,
            "scaler_scale": scaler.scale_,
            "feature_cols": feature_cols
        }
        torch.save(checkpoint, model_path)
        
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] - Train Loss: {t_loss:.4f}, Acc: {t_acc:.2%} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2%}")

print(f"Training completed. Checkpoint (weights + scaler) saved to '{model_path}'.")


## 6. Threshold Sensitivity Analysis & Out-of-Sample Backtest

We evaluate probabilistic predictions on the test set and sweep confidence thresholds $	heta \in [0.50, 0.51, 0.52, 0.53, 0.55]$.


In [ ]:
# Load checkpoint
checkpoint = torch.load(model_path, map_location=device)
best_model = TradingDNN(input_dim=len(feature_cols), hidden_units=[64, 32], dropout=0.0)
best_model.load_state_dict(checkpoint["state_dict"])
best_model.to(device).eval()

# Inference on Test Set
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    probs = torch.sigmoid(best_model(X_test_tensor)).cpu().numpy().flatten()

test_res = test_df.copy()
test_res['prob_up'] = probs
tc = 0.0005

# Threshold Sensitivity Sweep (matches Slide Deck table)
sweep_results = []
for theta in [0.50, 0.51, 0.52, 0.53, 0.55]:
    pos = pd.Series(0, index=test_res.index)
    pos[test_res['prob_up'] > theta] = 1
    pos[test_res['prob_up'] < (1.0 - theta)] = -1
    
    strat_gross = pos * test_res['target_return']
    trades = pos.diff().abs().fillna(pos.abs())
    strat_net = strat_gross - (trades * tc)
    
    ann_ret = np.exp(strat_net.sum() / (len(test_res) / 252)) - 1.0
    vol = strat_net.std() * np.sqrt(252)
    sharpe = ann_ret / vol if vol > 0 else 0.0
    sweep_results.append({
        "Threshold (theta)": f"{theta:.2f}",
        "Active Positions": f"{(pos != 0).mean():.1%}",
        "Switches": int(trades.sum()),
        "Ann. Return (Net)": f"{ann_ret:.2%}",
        "Sharpe (Net)": f"{sharpe:.2f}"
    })

print("--- THRESHOLD SENSITIVITY MATRIX ---")
print(pd.DataFrame(sweep_results).to_string(index=False))

# Selected threshold (theta = 0.52)
up_thresh = 0.52
down_thresh = 0.48
test_res['position'] = 0
test_res.loc[test_res['prob_up'] > up_thresh, 'position'] = 1
test_res.loc[test_res['prob_up'] < down_thresh, 'position'] = -1

test_res['strategy_gross'] = test_res['position'] * test_res['target_return']
test_res['trades'] = test_res['position'].diff().abs().fillna(test_res['position'].abs())
test_res['strategy_net'] = test_res['strategy_gross'] - (test_res['trades'] * tc)
test_res['creturns_market'] = np.exp(test_res['target_return'].cumsum())
test_res['creturns_net'] = np.exp(test_res['strategy_net'].cumsum())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(train_losses, label='Train Loss', color='#2F80ED')
axes[0].plot(val_losses, label='Val Loss', color='#EB5757')
axes[0].set_title('Cross-Entropy Loss Convergence')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(test_res['creturns_market'], label='Buy & Hold (SPY)', color='gray')
axes[1].plot(test_res['creturns_net'], label='PyTorch DNN (Net, theta=0.52)', color='#27AE60', linewidth=2)
axes[1].set_title('Out-of-Sample Strategy Equity Curve')
axes[1].set_ylabel('Cumulative Growth')
axes[1].legend()

plt.tight_layout()
plt.show()


---
### Summary & Transition to Session 3
- PyTorch on GPU allows rapid exploration of non-linear financial patterns.
- Model weights and feature scalers are serialized together into `best_trading_dnn.pt`.
- In **Session 3**, we package our trained model artifact into a complete **ZeroMQ streaming cloud trading system** with SQLite storage and live risk guardrails.
